In [4]:
import sys
print(sys.executable)


c:\Users\E100873\OneDrive - EUParks\Desktop\Internship\Hackathon tool\.venv\Scripts\python.exe


In [3]:
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt

BOOKING_START_DATE = "2023-01-01"
LAT = 52.0907   # Utrecht / Netherlands proxy
LON = 5.1214

DATA_DIR = Path("data")
BOOKING_FILE = DATA_DIR / "booking.csv"   # change if your filename differs

ModuleNotFoundError: No module named 'pandas'

In [ ]:
if not BOOKING_FILE.exists():
    raise FileNotFoundError(f"Could not find: {BOOKING_FILE.resolve()}")

bookings = pd.read_csv(BOOKING_FILE)

print("Columns found:")
print(bookings.columns.tolist())

bookings.head()

In [ ]:
bookings.columns = [c.strip().lower() for c in bookings.columns]

if "booking_date" not in bookings.columns:
    raise ValueError("Expected a 'booking_date' column in the booking CSV.")

bookings["booking_date"] = pd.to_datetime(bookings["booking_date"], errors="coerce")
bookings = bookings.dropna(subset=["booking_date"]).copy()

bookings = bookings[bookings["booking_date"] >= pd.Timestamp(BOOKING_START_DATE)]
bookings = bookings.sort_values("booking_date").reset_index(drop=True)

# Make sure numeric columns are numeric if present
numeric_cols = ["bookings", "rental_revenue", "guests", "booked_nights", "avg_adr"]
for col in numeric_cols:
    if col in bookings.columns:
        bookings[col] = pd.to_numeric(bookings[col], errors="coerce")

bookings.head()

In [ ]:
def fetch_historical_weather(lat: float, lon: float, start_date: str, end_date: str) -> pd.DataFrame:
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum",
        "timezone": "auto",
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    daily = data.get("daily", {})
    weather = pd.DataFrame({
        "date": pd.to_datetime(daily.get("time", []), errors="coerce"),
        "temp_max": daily.get("temperature_2m_max", []),
        "temp_min": daily.get("temperature_2m_min", []),
        "temp_mean": daily.get("temperature_2m_mean", []),
        "rain_mm": daily.get("precipitation_sum", []),
    })

    return weather

In [ ]:
end_date = bookings["booking_date"].max().strftime("%Y-%m-%d")

weather = fetch_historical_weather(
    lat=LAT,
    lon=LON,
    start_date=BOOKING_START_DATE,
    end_date=end_date,
)

weather.head()

In [ ]:
comparison = bookings.merge(
    weather,
    left_on="booking_date",
    right_on="date",
    how="inner"
)

comparison = comparison.drop(columns=["date"]).copy()

print("Merged shape:", comparison.shape)
comparison.head()

In [ ]:
cols_to_check = ["bookings", "temp_mean", "temp_max", "rain_mm"]
available = [c for c in cols_to_check if c in comparison.columns]

corr = comparison[available].corr(numeric_only=True)
corr

In [ ]:
if "bookings" in comparison.columns:
    if "temp_mean" in comparison.columns:
        print("Correlation between bookings and average temperature:",
              round(comparison["bookings"].corr(comparison["temp_mean"]), 3))

    if "rain_mm" in comparison.columns:
        print("Correlation between bookings and rainfall:",
              round(comparison["bookings"].corr(comparison["rain_mm"]), 3))

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(comparison["booking_date"], comparison["bookings"])
plt.title("Bookings over time")
plt.xlabel("Date")
plt.ylabel("Bookings")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(comparison["booking_date"], comparison["temp_mean"])
plt.title("Average temperature over time")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(comparison["temp_mean"], comparison["bookings"], alpha=0.5)
plt.title("Bookings vs Average Temperature")
plt.xlabel("Average Temperature (°C)")
plt.ylabel("Bookings")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(comparison["rain_mm"], comparison["bookings"], alpha=0.5)
plt.title("Bookings vs Rainfall")
plt.xlabel("Rainfall (mm)")
plt.ylabel("Bookings")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(comparison["rain_mm"], comparison["bookings"], alpha=0.5)
plt.title("Bookings vs Rainfall")
plt.xlabel("Rainfall (mm)")
plt.ylabel("Bookings")
plt.tight_layout()
plt.show()

In [ ]:
comparison["month"] = comparison["booking_date"].dt.to_period("M").dt.to_timestamp()

monthly = comparison.groupby("month", as_index=False).agg({
    "bookings": "mean",
    "temp_mean": "mean",
    "rain_mm": "mean"
})

monthly.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly["month"], monthly["bookings"], label="Avg bookings")
plt.plot(monthly["month"], monthly["temp_mean"], label="Avg temperature")
plt.title("Monthly average bookings and temperature")
plt.xlabel("Month")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
for lag in [0, 1, 2, 3, 7]:
    comparison[f"temp_mean_lag_{lag}"] = comparison["temp_mean"].shift(lag)
    comparison[f"rain_mm_lag_{lag}"] = comparison["rain_mm"].shift(lag)

lag_results = []
for lag in [0, 1, 2, 3, 7]:
    temp_corr = comparison["bookings"].corr(comparison[f"temp_mean_lag_{lag}"])
    rain_corr = comparison["bookings"].corr(comparison[f"rain_mm_lag_{lag}"])
    lag_results.append({
        "lag_days": lag,
        "temp_corr": temp_corr,
        "rain_corr": rain_corr,
    })

lag_df = pd.DataFrame(lag_results)
lag_df